In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
.appName("dzc26 batch") \
.getOrCreate()

In [4]:
spark.version

'4.0.2'

In [5]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-02-28 03:02:34--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 13.35.218.108, 13.35.218.193, 13.35.218.179, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|13.35.218.108|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  11.5MB/s    in 7.3s    

2026-02-28 03:02:42 (9.27 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [66]:
df = spark.read.parquet("yellow_tripdata_2025-11.parquet")

df.repartition(4).write.parquet("yellow_11")

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [7]:
!ls -la

total 69488
drwxr-xr-x 1 root root     4096 Feb 28 03:04 .
drwxr-xr-x 1 root root     4096 Feb 28 02:59 ..
drwxr-xr-x 4 root root     4096 Jan 16 14:24 .config
drwxr-xr-x 1 root root     4096 Jan 16 14:24 sample_data
drwxr-xr-x 2 root root     4096 Feb 28 03:04 yellow_11
-rw-r--r-- 1 root root 71134255 Dec 19 15:51 yellow_tripdata_2025-11.parquet


In [10]:
! cd yellow_11/ && ls -l

total 103800
-rw-r--r-- 1 root root 26583767 Feb 28 03:04 part-00000-c8432d62-50be-4800-847b-e9046b8095e3-c000.snappy.parquet
-rw-r--r-- 1 root root 26562682 Feb 28 03:04 part-00001-c8432d62-50be-4800-847b-e9046b8095e3-c000.snappy.parquet
-rw-r--r-- 1 root root 26558802 Feb 28 03:04 part-00002-c8432d62-50be-4800-847b-e9046b8095e3-c000.snappy.parquet
-rw-r--r-- 1 root root 26573748 Feb 28 03:04 part-00003-c8432d62-50be-4800-847b-e9046b8095e3-c000.snappy.parquet
-rw-r--r-- 1 root root        0 Feb 28 03:04 _SUCCESS


In [11]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [61]:
from pyspark.sql.functions import to_date, col, timestamp_diff, datepart, desc, asc

In [15]:
df.filter(to_date(df.tpep_pickup_datetime) == "2025-11-15").count()

162604

In [47]:
df1 = df.withColumn(
    "trip_duration_hours",
    timestamp_diff("HOUR", col("tpep_pickup_datetime"), col("tpep_dropoff_datetime"))
    ).select("tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_duration_hours")

In [50]:
df1.orderBy(col("trip_duration_hours").desc()).show(5)

+--------------------+---------------------+-------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_duration_hours|
+--------------------+---------------------+-------------------+
| 2025-11-26 20:22:12|  2025-11-30 15:01:00|                 90|
| 2025-11-03 10:42:55|  2025-11-06 14:55:45|                 76|
| 2025-11-27 04:22:41|  2025-11-30 09:19:35|                 76|
| 2025-11-07 11:23:22|  2025-11-10 08:40:41|                 69|
| 2025-11-18 17:12:47|  2025-11-21 12:17:37|                 67|
+--------------------+---------------------+-------------------+
only showing top 5 rows


In [51]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-02-28 03:34:37--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.169.132.39, 3.169.132.186, 3.169.132.219, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.169.132.39|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-02-28 03:34:38 (118 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [75]:
location = spark.read.csv("taxi_zone_lookup.csv", header=True)

In [55]:
df2 = df.select("PULocationID").groupBy("PULocationID").count()

In [62]:
df2.show()

+------------+-----+
|PULocationID|count|
+------------+-----+
|         148|51711|
|         243| 4901|
|          31|   89|
|         137|46493|
|          85| 1711|
|         251|   12|
|          65| 5394|
|         255| 9141|
|          53|  307|
|         133|  985|
|          78| 1129|
|         155|  913|
|         108|  515|
|         211|34737|
|          34|  947|
|         193| 3199|
|         115|   42|
|         126|  778|
|         101|  271|
|          81|  711|
+------------+-----+
only showing top 20 rows


In [74]:
df2 = spark.read.parquet("yellow_tripdata_2025-11.parquet").groupBy("PULocationID").count()
df2.show()

+------------+-----+
|PULocationID|count|
+------------+-----+
|         148|51711|
|         243| 4901|
|          31|   89|
|         137|46493|
|          85| 1711|
|         251|   12|
|          65| 5394|
|         255| 9141|
|          53|  307|
|         133|  985|
|          78| 1129|
|         155|  913|
|         108|  515|
|         211|34737|
|          34|  947|
|         193| 3199|
|         115|   42|
|         126|  778|
|         101|  271|
|          81|  711|
+------------+-----+
only showing top 20 rows


In [76]:
df2 \
.join(location, df2.PULocationID == location.LocationID) \
.select("PULocationID", "Zone", col("count").alias("Frequency")) \
.orderBy(asc("Frequency")) \
.show(truncate=False)

+------------+---------------------------------------------+---------+
|PULocationID|Zone                                         |Frequency|
+------------+---------------------------------------------+---------+
|105         |Governor's Island/Ellis Island/Liberty Island|1        |
|5           |Arden Heights                                |1        |
|84          |Eltingville/Annadale/Prince's Bay            |1        |
|187         |Port Richmond                                |3        |
|204         |Rossville/Woodrow                            |4        |
|199         |Rikers Island                                |4        |
|111         |Green-Wood Cemetery                          |4        |
|109         |Great Kills                                  |4        |
|2           |Jamaica Bay                                  |5        |
|251         |Westerleigh                                  |12       |
|59          |Crotona Park                                 |14       |
|176  